# 1. INTRODUCCIÓN

(...)

# 2. EXTRACCIÓN DE DATOS

In [16]:
import os
import requests
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta
import pandas as pd

load_dotenv()
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

CRYPTOS = [
    { "name": "Bitcoin", "binance": "BTCUSDT", "coingecko": "bitcoin" },
    { "name": "Ethereum", "binance": "ETHUSDT", "coingecko": "ethereum" },
    { "name": "Solana", "binance": "SOLUSDT", "coingecko": "solana" },
    { "name": "BinanceCoin", "binance": "BNBUSDT", "coingecko": "binancecoin" },
    { "name": "Ripple", "binance": "XRPUSDT", "coingecko": "ripple" },
]

EXTRACTION_CONFIG = {
    "binance": {
        "url": "https://api.binance.com/api/v3/klines",
        "params": {
            "interval": "1h",
            "startTime": int((datetime.now() - timedelta(days=365)).timestamp() * 1000),
            "endTime": int(datetime.now().timestamp() * 1000)
        }
    },
    "coingecko": {
        "url": "https://api.coingecko.com/api/v3/coins/{id}/market_chart",
        "params": {
            "vs_currency": "usd",
            "days": 365,
            "x_cg_demo_api_key": COINGECKO_API_KEY
        }
    }
}

def extract_binance_full_history(symbol, start_time, end_time, interval="1h"):
    """Paginación para obtener más de 1000 velas de Binance"""
    url = EXTRACTION_CONFIG["binance"]["url"]
    all_data = []
    limit = 1000
    current_start = start_time

    while current_start < end_time:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start,
            "endTime": end_time,
            "limit": limit
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if not data:
            break

        all_data.extend(data)

        # siguiente batch (última vela + 1ms)
        current_start = data[-1][0] + 1

        time.sleep(0.2)  # evitar rate limit

    return all_data


def extract_data(url, params):
    try:
        response = requests.get(url=url, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print("Ha ocurrido un error: ", e)


dfs_cryptos = {}

for crypto in CRYPTOS:

    crypto_name = crypto["name"]
    crypto_symbol_binance = crypto["binance"]
    crypto_symbol_coingecko = crypto["coingecko"]

    print(f"---- Procesando {crypto_name}... ----\n")

    # ---------------- BINANCE (PAGINADO) ----------------
    data_binance = extract_binance_full_history(
        crypto_symbol_binance,
        EXTRACTION_CONFIG["binance"]["params"]["startTime"],
        EXTRACTION_CONFIG["binance"]["params"]["endTime"]
    )

    df_binance = pd.DataFrame(data_binance, columns=[
        "open_time", "open_price", "high_price", "low_price",
        "close_price", "volume", "close_time",
        "quote_asset_volume", "number_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "unused_field"
    ])

    df_binance = df_binance[
        ["open_time", "open_price", "high_price", "low_price",
         "close_price", "volume", "close_time"]
    ]

    df_binance["open_time"] = pd.to_datetime(df_binance["open_time"], unit="ms")


    # ---------------- COINGECKO ----------------
    coingecko_url = EXTRACTION_CONFIG["coingecko"]["url"].format(
        id=crypto_symbol_coingecko
    )

    data_coingecko = extract_data(coingecko_url, EXTRACTION_CONFIG["coingecko"]["params"])
    data_coingecko = data_coingecko["market_caps"]

    df_coingecko = pd.DataFrame(data_coingecko, columns=["open_time", "market_cap"])
    df_coingecko["open_time"] = pd.to_datetime(df_coingecko["open_time"], unit="ms")

    df_coingecko = df_coingecko.set_index("open_time").sort_index()
    df_coingecko = df_coingecko.resample("1h").ffill()


    # ---------------- ALINEACIÓN ----------------
    start = max(df_binance["open_time"].min(), df_coingecko.index.min())
    end = min(df_binance["open_time"].max(), df_coingecko.index.max())

    df_binance = df_binance[
        (df_binance["open_time"] >= start) &
        (df_binance["open_time"] <= end)
    ]

    df_coingecko = df_coingecko.loc[start:end]

    df_final = df_binance.merge(
        df_coingecko,
        left_on="open_time",
        right_index=True,
        how="left"
    )

    df_final["crypto"] = crypto_name
    df_final = df_final[["crypto"] + [col for col in df_final.columns if col != "crypto"]] # Ordenar para que el nombre de la crypto sea la primera columna
    dfs_cryptos[crypto_name] = df_final
    print(f"---- Obtención de datos de {crypto_name} ✅ ----\n")


print("---- 🧪 DATASETS FINALES 🧪 ----\n")

for name, df_crypto in dfs_cryptos.items():
    print(f"---- Datos finales obtenidos para el DataFrame de: {name} ----\n")
    display(df_crypto.head(10)) 
    print(df_crypto.shape)

---- Procesando Bitcoin... ----

---- Obtención de datos de Bitcoin ✅ ----

---- Procesando Ethereum... ----

---- Obtención de datos de Ethereum ✅ ----

---- Procesando Solana... ----

---- Obtención de datos de Solana ✅ ----

---- Procesando BinanceCoin... ----

---- Obtención de datos de BinanceCoin ✅ ----

---- Procesando Ripple... ----

---- Obtención de datos de Ripple ✅ ----

---- 🧪 DATASETS FINALES 🧪 ----

---- Datos finales obtenidos para el DataFrame de: Bitcoin ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
2,Bitcoin,2025-04-13 00:00:00,85276.91000000,85571.12000000,85012.00000000,85495.72000000,519.16586000,1744505999999,1.693601e+12
3,Bitcoin,2025-04-13 01:00:00,85495.72000000,86100.00000000,85142.46000000,85259.77000000,1921.79914000,1744509599999,1.693601e+12
4,Bitcoin,2025-04-13 02:00:00,85259.76000000,85451.85000000,84724.91000000,85302.22000000,1987.96886000,1744513199999,1.693601e+12
5,Bitcoin,2025-04-13 03:00:00,85302.21000000,85724.79000000,85247.03000000,85432.31000000,1663.06089000,1744516799999,1.693601e+12
6,Bitcoin,2025-04-13 04:00:00,85432.31000000,85436.68000000,84464.44000000,84619.98000000,2229.02689000,1744520399999,1.693601e+12
7,Bitcoin,2025-04-13 05:00:00,84619.98000000,84804.57000000,84399.42000000,84515.58000000,2128.82276000,1744523999999,1.693601e+12
8,Bitcoin,2025-04-13 06:00:00,84515.57000000,84849.06000000,84444.72000000,84608.69000000,449.65847000,1744527599999,1.693601e+12
9,Bitcoin,2025-04-13 07:00:00,84608.70000000,84769.87000000,84413.09000000,84413.10000000,521.59067000,1744531199999,1.693601e+12
10,Bitcoin,2025-04-13 08:00:00,84413.10000000,84693.20000000,84277.03000000,84664.60000000,526.53348000,1744534799999,1.693601e+12
11,Bitcoin,2025-04-13 09:00:00,84664.61000000,84948.25000000,84504.58000000,84918.24000000,465.84452000,1744538399999,1.693601e+12


(8758, 9)
---- Datos finales obtenidos para el DataFrame de: Ethereum ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
2,Ethereum,2025-04-13 00:00:00,1644.19000000,1649.49000000,1634.77000000,1642.10000000,13731.74890000,1744505999999,1.986964e+11
3,Ethereum,2025-04-13 01:00:00,1642.09000000,1649.75000000,1609.09000000,1617.11000000,24993.83030000,1744509599999,1.986964e+11
4,Ethereum,2025-04-13 02:00:00,1617.11000000,1628.44000000,1606.55000000,1625.40000000,24604.68170000,1744513199999,1.986964e+11
5,Ethereum,2025-04-13 03:00:00,1625.41000000,1629.23000000,1621.61000000,1624.53000000,12047.01730000,1744516799999,1.986964e+11
6,Ethereum,2025-04-13 04:00:00,1624.52000000,1624.66000000,1602.71000000,1609.00000000,21497.81090000,1744520399999,1.986964e+11
7,Ethereum,2025-04-13 05:00:00,1608.99000000,1613.52000000,1597.47000000,1609.28000000,18123.42230000,1744523999999,1.986964e+11
8,Ethereum,2025-04-13 06:00:00,1609.29000000,1623.32000000,1607.89000000,1616.99000000,9954.27540000,1744527599999,1.986964e+11
9,Ethereum,2025-04-13 07:00:00,1616.99000000,1621.41000000,1610.96000000,1613.50000000,7664.78400000,1744531199999,1.986964e+11
10,Ethereum,2025-04-13 08:00:00,1613.51000000,1621.94000000,1608.80000000,1620.60000000,9737.61110000,1744534799999,1.986964e+11
11,Ethereum,2025-04-13 09:00:00,1620.60000000,1627.81000000,1617.04000000,1627.14000000,10629.19500000,1744538399999,1.986964e+11


(8758, 9)
---- Datos finales obtenidos para el DataFrame de: Solana ----



...

(8758, 9)
---- Datos finales obtenidos para el DataFrame de: BinanceCoin ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
2,BinanceCoin,2025-04-13 00:00:00,597.30000000,597.36000000,595.11000000,596.91000000,3438.38400000,1744505999999,8.708992e+10
3,BinanceCoin,2025-04-13 01:00:00,596.91000000,598.67000000,593.70000000,594.46000000,4635.40600000,1744509599999,8.708992e+10
4,BinanceCoin,2025-04-13 02:00:00,594.46000000,595.33000000,593.13000000,595.01000000,3396.77100000,1744513199999,8.708992e+10
5,BinanceCoin,2025-04-13 03:00:00,595.00000000,596.34000000,594.32000000,595.90000000,4241.83900000,1744516799999,8.708992e+10
6,BinanceCoin,2025-04-13 04:00:00,595.90000000,595.93000000,591.51000000,592.30000000,7911.81900000,1744520399999,8.708992e+10
7,BinanceCoin,2025-04-13 05:00:00,592.29000000,593.50000000,590.90000000,592.82000000,4218.65300000,1744523999999,8.708992e+10
8,BinanceCoin,2025-04-13 06:00:00,592.82000000,594.10000000,592.75000000,592.80000000,3418.40900000,1744527599999,8.708992e+10
9,BinanceCoin,2025-04-13 07:00:00,592.80000000,594.00000000,592.00000000,593.33000000,3000.01600000,1744531199999,8.708992e+10
10,BinanceCoin,2025-04-13 08:00:00,593.34000000,593.88000000,592.40000000,593.75000000,2563.82500000,1744534799999,8.708992e+10
11,BinanceCoin,2025-04-13 09:00:00,593.75000000,594.38000000,592.62000000,594.36000000,3903.38700000,1744538399999,8.708992e+10


(8758, 9)
---- Datos finales obtenidos para el DataFrame de: Ripple ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
2,Ripple,2025-04-13 00:00:00,2.15970000,2.17350000,2.14300000,2.16070000,5685215.00000000,1744505999999,1.258237e+11
3,Ripple,2025-04-13 01:00:00,2.16070000,2.17400000,2.14400000,2.15100000,4808792.00000000,1744509599999,1.258237e+11
4,Ripple,2025-04-13 02:00:00,2.15110000,2.15730000,2.13500000,2.15090000,5050225.00000000,1744513199999,1.258237e+11
5,Ripple,2025-04-13 03:00:00,2.15100000,2.16150000,2.14210000,2.15630000,3626246.00000000,1744516799999,1.258237e+11
6,Ripple,2025-04-13 04:00:00,2.15630000,2.15970000,2.13530000,2.13820000,4246369.00000000,1744520399999,1.258237e+11
7,Ripple,2025-04-13 05:00:00,2.13830000,2.14530000,2.12680000,2.13750000,3957048.00000000,1744523999999,1.258237e+11
8,Ripple,2025-04-13 06:00:00,2.13760000,2.14880000,2.13200000,2.14150000,2304882.00000000,1744527599999,1.258237e+11
9,Ripple,2025-04-13 07:00:00,2.14150000,2.15420000,2.13860000,2.14630000,2818641.00000000,1744531199999,1.258237e+11
10,Ripple,2025-04-13 08:00:00,2.14640000,2.18060000,2.13510000,2.17610000,8166231.00000000,1744534799999,1.258237e+11
11,Ripple,2025-04-13 09:00:00,2.17620000,2.23180000,2.16940000,2.22990000,13738427.00000000,1744538399999,1.258237e+11


(8758, 9)


# 3. TRANSFORMACIÓN DE DATOS

In [ ]:
# Transform:
## Limpieza de nulos, duplicados y conversión de tipos
## Normalización estructural

# 4. ALMACENAMIENTO DE DATOS (PARQUET)

In [ ]:
# Load:
## Guardar en .parquet
## 1 fichero por CRYPTO